# Missing Data

Calculate how many days per study id should be collected and compare with how many days are actually there

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

notebook_dir = Path().resolve()
sys.path.append(str(notebook_dir.parent))        # ../
sys.path.append(str(notebook_dir.parent.parent)) # ../../


import pandas as pd
from datetime import datetime

from plotly.subplots import make_subplots
import plotly.graph_objects as go



from data_helper.clean_data_functions import clean_data_baseline_optimized, clean_data_no_data_days_optimized
from plot_helper.colors import COLORS


#create empty dataframe
df_row_counts = pd.DataFrame()


## Plot Function

In [ ]:
def plot_actual_vs_theoretical(
    df,
    type_col='type',
    id_col='study_id',
    actual_col='days_actual',
    theoretical_col='days_theoretical',
    types=('MHT', 'Dual/GLP1-RA'),
    thresholds=(7, 7),
    height=800,
    title="Title"
):
    colors = {
        'actual': COLORS.get("blue1"),
        'theoretical': COLORS.get("orange1"),
        'threshold': COLORS.get("green1")
    }

    fig = make_subplots(
        rows=len(types), cols=1,
        subplot_titles=list(types)
    )

    for i, (treatment_type, threshold) in enumerate(zip(types, thresholds), start=1):
        df_type = df[df[type_col] == treatment_type]
        show_legend = (i == 1)

        fig.add_trace(go.Bar(
            x=df_type[id_col],
            y=df_type[actual_col],
            name='Actual',
            marker_color=colors['actual'],
            offsetgroup='actual',
            showlegend=show_legend,
        ), row=i, col=1)

        fig.add_trace(go.Bar(
            x=df_type[id_col],
            y=df_type[theoretical_col],
            name='Theoretical',
            marker_color=colors['theoretical'],
            offsetgroup='theoretical',
            showlegend=show_legend,
        ), row=i, col=1)

        fig.add_hline(
            y=threshold,
            line_color=colors['threshold'],
            row=i, col=1,
            showlegend=show_legend,
            name='Threshold',
        )

    fig.update_layout(barmode='group', height=height, title_text=title)
    return fig

## Calculate Theoretical Days per Study_Id

In [ ]:
#get start and end visit date from visit_dates.csv
df_visit_dates = pd.read_csv("../../data/checks/visit_dates_2026-07-08.csv")
df_visit_dates = df_visit_dates[df_visit_dates['visit_name'] == 'V1']

#calculate nights for visit period
df_visit_dates['nights'] = (pd.to_datetime(df_visit_dates['end_visit_date']) - pd.to_datetime(df_visit_dates['start_visit_date'])).dt.days
df_visit_dates['days'] = (pd.to_datetime(df_visit_dates['end_visit_date']) - pd.to_datetime(df_visit_dates['start_visit_date'])).dt.days + 1

#add if 1b or 1a (if days >= 20 type is MHT else Dual/GLP1-RA)
df_visit_dates['type'] = 'MHT'
df_visit_dates.loc[df_visit_dates['days'] < 20, 'type'] = 'Dual/GLP1-RA'

display(df_visit_dates.head())
display(df_visit_dates[df_visit_dates['study_id']=='DEC_42'])

## Calculate Actual Days per Study_Id

### Garmin Connect Sleep Summary

In [ ]:
#Get downloaded data
folder_sleep_summary = Path("../../data/sleep_summary")

#Get merged csv file paths
sleep_summary_files = [f.path for f in os.scandir(folder_sleep_summary) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_sleep_summary_raw = pd.concat(
    [pd.read_csv(f) for f in sleep_summary_files],
    ignore_index=True
)


#clean dataframes
##match baseline dates
print(f"Sleep summary data before first cropping: {df_sleep_summary_raw.shape}")
df_sleep_summary_raw = clean_data_baseline_optimized([df_sleep_summary_raw])[0]
print(f"Sleep summary data after first cropping: {df_sleep_summary_raw.shape}")
print(f"------"*20)

#*--Make sure sleep does not start too early as calendarDate == wake up date
#*drop rows where calendarDate is smaller than start visit date plus 1 day or calendarDate is larger than end visit date
df_visit_dates_v1 = pd.read_csv("../../data/checks/visit_dates_2026-07-08.csv")
df_visit_dates_v1 = df_visit_dates_v1[df_visit_dates_v1['visit_name'] == 'V1'].copy()

# Convert dates
df_sleep_summary_raw["calendarDate"] = pd.to_datetime(
    df_sleep_summary_raw["calendarDate"],
    errors="coerce"
)

df_visit_dates_v1["start_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["start_visit_date"],
    errors="coerce"
)

df_visit_dates_v1["end_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["end_visit_date"],
    errors="coerce"
)
# Keep only needed visit columns
df_visit_dates_v1 = df_visit_dates_v1[
    ["study_id", "start_visit_date", "end_visit_date"]
].drop_duplicates("study_id")

# Merge visit dates onto sleep summary
df_sleep_summary_raw = df_sleep_summary_raw.merge(
    df_visit_dates_v1,
    on="study_id",
    how="left"
)

start_cutoff = df_sleep_summary_raw["start_visit_date"] + pd.Timedelta(days=1)

mask_keep = (
    (df_sleep_summary_raw["calendarDate"] >= start_cutoff) &
    (df_sleep_summary_raw["calendarDate"] <= df_sleep_summary_raw["end_visit_date"])
)
df_sleep_summary_raw = df_sleep_summary_raw[mask_keep].copy()

print(f"Sleep summary data after second cropping: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print("------"*20)

##drop days with no data
df_sleep_summary_raw = clean_data_no_data_days_optimized([df_sleep_summary_raw])[0]
print(f"Sleep summary data after dropping days with no data: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print("------"*20)

#count nights per study id
df_sleep_summary = df_sleep_summary_raw[['calendarDate', 'study_id']]
df_sleep_summary_counts = df_sleep_summary.groupby(["study_id"]).count().reset_index()
df_sleep_summary_counts = df_sleep_summary_counts.rename(columns={'calendarDate': 'nights'})
display(df_sleep_summary_counts)



In [ ]:
#merge actual with theoretical nights
df_sleep_summary_merge = df_sleep_summary_counts[['study_id', 'nights']].merge(df_visit_dates[['study_id', 'nights', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_sleep_summary_merge)


#total rows theoretically
total_theoretically = df_sleep_summary_merge['nights_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_sleep_summary_merge['nights_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_sleep_summary_merge['study_id'].nunique()
df_sleep_summary_row_counts = pd.DataFrame([{
    'source': 'garmin-connect-sleep-summary', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically,
    'missing_rows': total_theoretically - total_actual, 
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_sleep_summary_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_sleep_summary_bad = df_sleep_summary_merge[df_sleep_summary_merge['nights_actual']!= df_sleep_summary_merge['nights_theoretical']]

#plot actual vs theoretical
fig_f1 = plot_actual_vs_theoretical(
    df_sleep_summary_bad,
    actual_col='nights_actual',
    theoretical_col='nights_theoretical',
    title='Garmin Connect Sleep Summary'
)
fig_f1.show()


#store image and csv
#if there already exists a file with same name move it to subfolder "archive"
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')
files = os.listdir('../../output/0_missing_data/initial_check/')
for file in files:
    if file.startswith('connect_sleep_summary_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-connect-sleep-summary_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
fig_f1.write_image(f"../../output/0_missing_data/initial_check/connect_sleep_summary_{date}.png", scale=2, width=1200, height=800)
df_sleep_summary_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-connect-sleep-summary_{date}.csv', index=False)


### Garmin Connect Sleep Stage

In [ ]:
#Get downloaded data
folder_sleep_stage = Path("../../data/sleep_stage")

#Get merged csv file paths
sleep_stage_files = [f.path for f in os.scandir(folder_sleep_stage) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_sleep_stage_raw = pd.concat(
    [pd.read_csv(f) for f in sleep_stage_files],
    ignore_index=True
)
df_sleep_stage_raw.head()

#clean dataframes
##match baseline dates
print(f"Sleep stage data before first cropping: {df_sleep_stage_raw.shape}")
df_sleep_stage_raw = clean_data_baseline_optimized([df_sleep_stage_raw])[0]
print(f"Sleep stage data after first cropping: {df_sleep_stage_raw.shape}")
print(f"------"*20)


##drop days with no data
df_sleep_stage_raw = clean_data_no_data_days_optimized([df_sleep_stage_raw])[0]
print(f"Sleep stage data after dropping days with no data: {df_sleep_stage_raw.shape}")

print("------"*20)

#count days per study id
df_sleep_stage = df_sleep_stage_raw[['datetime_utc', 'timezone', 'datetime', 'study_id']]
dt_utc = pd.to_datetime(
            df_sleep_stage["datetime_utc"],
            errors="coerce",
            utc=True
        )
df_sleep_stage['datetime'] = [
            ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
            for ts, tz in zip(dt_utc, df_sleep_stage["timezone"])
        ]
df_sleep_stage = df_sleep_stage.drop(columns=['datetime_utc', 'timezone'])
display(df_sleep_stage.head())

#only keep unique datetime per study id
df_sleep_stage = df_sleep_stage.drop_duplicates()
df_sleep_stage_counts = df_sleep_stage.groupby(["study_id"]).count().reset_index()
df_sleep_stage_counts = df_sleep_stage_counts.rename(columns={'datetime': 'days'})
display(df_sleep_stage_counts)



In [ ]:
#merge actual with theoretical days
df_sleep_stage_merge = df_sleep_stage_counts[['study_id', 'days']].merge(df_visit_dates[['study_id', 'days', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_sleep_stage_merge)


#total rows theoretically
total_theoretically = df_sleep_stage_merge['days_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_sleep_stage_merge['days_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_sleep_stage_merge['study_id'].nunique()
df_sleep_stage_row_counts = pd.DataFrame([{
    'source': 'garmin-connect-sleep-stage', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows': total_theoretically - total_actual,
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_sleep_stage_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_sleep_stage_bad = df_sleep_stage_merge[df_sleep_stage_merge['days_actual']!= df_sleep_stage_merge['days_theoretical']]

#plot theoretical vs actual
fig_f2 = plot_actual_vs_theoretical(
    df_sleep_stage_bad,
    actual_col='days_actual',
    theoretical_col='days_theoretical',
    title='Garmin Connect Sleep Stage'
)
fig_f2.show()
fig_f1 = make_subplots(
    rows=2, cols=1,
    specs=[
        [{}], [{}]                 
    ],
    subplot_titles=(
    f"MHT",
    f"Dual/GLP1-RA",

))

#store missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('connect_sleep_stage_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-connect-sleep-stage_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
fig_f2.write_image(f"../../output/0_missing_data/initial_check/connect_sleep_stage_{date}.png", scale=2, width=1200, height=800)
df_sleep_stage_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-connect-sleep-stage_{date}.csv', index=False)

### Garmin Connect Epoch

In [ ]:
#Get downloaded data
folder_epoch = Path("../../data/epoch")

#Get merged csv file paths
epoch_files = [f.path for f in os.scandir(folder_epoch) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_epoch_raw = pd.concat(
    [pd.read_csv(f) for f in epoch_files],
    ignore_index=True
)

dt_utc_epoch = pd.to_datetime(
            df_epoch_raw["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_epoch_raw['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_epoch, df_epoch_raw["timezone"])
]


#clean dataframes
##match baseline dates
print(f"Epoch data before cropping: {df_epoch_raw.shape}")
df_epoch_raw = clean_data_baseline_optimized([df_epoch_raw])[0]
print(f"Epoch data after cropping: {df_epoch_raw.shape}")
display(df_epoch_raw.head())
print("------"*20)
##drop days with no data
df_epoch_raw = clean_data_no_data_days_optimized([df_epoch_raw])[0]
print(f"Epoch data after dropping days with no data: {df_epoch_raw.shape}")
display(df_epoch_raw.head())
print("------"*20)


#count days per study id
df_epoch = df_epoch_raw[['calendarDate', 'study_id']]
df_epoch = df_epoch.drop_duplicates()
df_epoch_counts = df_epoch.groupby(["study_id"]).count().reset_index()
df_epoch_counts = df_epoch_counts.rename(columns={'calendarDate': 'days'})
display(df_epoch_counts)



In [ ]:
#merge actual with theoretical days
df_epoch_merge = df_epoch_counts[['study_id', 'days']].merge(df_visit_dates[['study_id', 'days', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_epoch_merge)


#total rows theoretically
total_theoretically = df_epoch_merge['days_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_epoch_merge['days_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_epoch_merge['study_id'].nunique()
df_epoch_row_counts = pd.DataFrame([{
    'source': 'garmin-connect-epoch', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows': total_theoretically - total_actual,
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_epoch_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_epoch_bad = df_epoch_merge[df_epoch_merge['days_actual']!= df_epoch_merge['days_theoretical']]

#plot theoretical vs actual
fig_f3 = plot_actual_vs_theoretical(
    df_epoch_bad,
    title='Garmin Connect Epoch',
)
fig_f3.show()

#store missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('connect_epoch_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-connect-epoch_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
fig_f3.write_image(f"../../output/0_missing_data/initial_check/connect_epoch_{date}.png", scale=2, width=1200, height=800)
df_epoch_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-connect-epoch_{date}.csv', index=False)

### Garmin Device Steps

In [ ]:
#Get downloaded data
folder_device_step = Path("../../data/device_step")

#Get merged csv file paths
step_files = [f.path for f in os.scandir(folder_device_step) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_step_raw = pd.concat(
    [pd.read_csv(f) for f in step_files],
    ignore_index=True
)

dt_utc_step = pd.to_datetime(
            df_step_raw["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_step_raw['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_step, df_step_raw["timezone"])
]


#clean dataframes
##match baseline dates
print(f"Step data before cropping: {df_step_raw.shape}")
df_step_raw = clean_data_baseline_optimized([df_step_raw])[0]
print(f"Step data after cropping: {df_step_raw.shape}")
display(df_step_raw.head())
print("------"*20)
##drop days with no data
df_step_raw = clean_data_no_data_days_optimized([df_step_raw])[0]
print(f"Step data after dropping days with no data: {df_step_raw.shape}")
display(df_step_raw.head())
print("------"*20)

#count days per study id
df_step = df_step_raw[['calendarDate', 'study_id']]
df_step = df_step.drop_duplicates()
df_step_counts = df_step.groupby(["study_id"]).count().reset_index()
df_step_counts = df_step_counts.rename(columns={'calendarDate': 'days'})
display(df_step_counts)



In [ ]:
# #add DEC_46 as row with 0 days (as it is missing in the data)
# df_step_counts = pd.concat([df_step_counts, pd.DataFrame([{'study_id': 'DEC_46', 'days': 0}])], ignore_index=True)
#merge actual with theoretical days
df_step_merge = df_step_counts[['study_id', 'days']].merge(df_visit_dates[['study_id', 'days', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_step_merge)


#total rows theoretically
total_theoretically = df_step_merge['days_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_step_merge['days_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_step_merge['study_id'].nunique()
df_step_row_counts = pd.DataFrame([{
    'source': 'garmin-device-step', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows': total_theoretically - total_actual,
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_step_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_step_bad = df_step_merge[df_step_merge['days_actual']!= df_step_merge['days_theoretical']]

#plot theoretical vs actual
fig_f4 = plot_actual_vs_theoretical(
    df_step_bad,
    title='Garmin Device Step'
)
fig_f4.show()

#store missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('device_step_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-device-step_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
        
date = datetime.now().strftime("%Y-%m-%d")
fig_f4.write_image(f"../../output/0_missing_data/initial_check/device_step_{date}.png", scale=2, width=1200, height=800)
df_step_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-device-step_{date}.csv', index=False)

### Garmin Connect Daily Summary

In [ ]:
#Get downloaded data
folder_daily_summary = Path("../../data/daily_summary")

#Get merged csv file paths
daily_summary_files = [f.path for f in os.scandir(folder_daily_summary) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_daily_summary_raw = pd.concat(
    [pd.read_csv(f) for f in daily_summary_files],
    ignore_index=True
)

dt_utc_daily_summary = pd.to_datetime(
            df_daily_summary_raw["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_daily_summary_raw['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_daily_summary, df_daily_summary_raw["timezone"])
]


#clean dataframes
##match baseline dates
print(f"Daily summary data before cropping: {df_daily_summary_raw.shape}")
df_daily_summary_raw = clean_data_baseline_optimized([df_daily_summary_raw])[0]
print(f"Daily summary data after cropping: {df_daily_summary_raw.shape}")
display(df_daily_summary_raw.head())
print("------"*20)
##drop days with no data
df_daily_summary_raw = clean_data_no_data_days_optimized([df_daily_summary_raw])[0]
print(f"Daily summary data after dropping days with no data: {df_daily_summary_raw.shape}")
display(df_daily_summary_raw.head())
print("------"*20)


#count days per study id
df_daily_summary = df_daily_summary_raw[['calendarDate', 'study_id']]
df_daily_summary = df_daily_summary.drop_duplicates()
df_daily_summary_counts = df_daily_summary.groupby(["study_id"]).count().reset_index()
df_daily_summary_counts = df_daily_summary_counts.rename(columns={'calendarDate': 'days'})
display(df_daily_summary_counts)



In [ ]:
#merge actual with theoretical days
df_daily_summary_merge = df_daily_summary_counts[['study_id', 'days']].merge(df_visit_dates[['study_id', 'days', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_daily_summary_merge)


#total rows theoretically
total_theoretically = df_daily_summary_merge['days_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_daily_summary_merge['days_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_daily_summary_merge['study_id'].nunique()
df_daily_summary_row_counts = pd.DataFrame([{
    'source': 'garmin-connect-daily-summary', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows': total_theoretically - total_actual,
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_daily_summary_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_daily_summary_bad = df_daily_summary_merge[df_daily_summary_merge['days_actual']!= df_daily_summary_merge['days_theoretical']]

#plot theoretical vs actual
fig_f5 = plot_actual_vs_theoretical(
    df_daily_summary_bad,
    title='Garmin Connect Daily Summary'
)
fig_f5.show()

#store missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('connect_daily_summary_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-connect-daily-summary_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
fig_f5.write_image(f"../../output/0_missing_data/initial_check/connect_daily_summary_{date}.png", scale=2, width=1200, height=800)
df_daily_summary_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-connect-daily-summary_{date}.csv', index=False)

### Garmin Device BBI

In [ ]:
#Get downloaded data
folder_device_bbi = Path("../../data/bbi_device")

#Get merged csv file paths
bbi_files = [f.path for f in os.scandir(folder_device_bbi) if f.is_file() and f.name.endswith(".csv")]

#split into 4 lists
bbi_files_1 = bbi_files[0:len(bbi_files)//4]
bbi_files_2 = bbi_files[len(bbi_files)//4:len(bbi_files)//2]
bbi_files_3 = bbi_files[len(bbi_files)//2:3*len(bbi_files)//4]
bbi_files_4 = bbi_files[3*len(bbi_files)//4:len(bbi_files)]

#check if all files together are the same as bbi_files
bbi_check = bbi_files_1 + bbi_files_2 + bbi_files_3 + bbi_files_4
if set(bbi_check) == set(bbi_files):
    print("All files accounted for in the split.")
else:
    print("Some files are missing in the split.")
    print("Missing files: ", set(bbi_files) - set(bbi_check))

#create dataframes
df_bbi_raw_1 = pd.concat(
    [pd.read_csv(f) for f in bbi_files_1],
    ignore_index=True
)
dt_utc_bbi = pd.to_datetime(
            df_bbi_raw_1["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_bbi_raw_1['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_bbi, df_bbi_raw_1["timezone"])
]


#clean dataframes
##match baseline dates
print(f"BBI data before cropping: {df_bbi_raw_1.shape}")
df_bbi_raw_1 = clean_data_baseline_optimized([df_bbi_raw_1])[0]
print(f"BBI data after cropping: {df_bbi_raw_1.shape}")
display(df_bbi_raw_1.head())
print("------"*20)
##drop days with no data
df_bbi_raw_1 = clean_data_no_data_days_optimized([df_bbi_raw_1])[0]
print(f"BBI data after dropping days with no data: {df_bbi_raw_1.shape}")
display(df_bbi_raw_1.head())
print("------"*20)

#count days per study id
df_bbi_1 = df_bbi_raw_1[['calendarDate', 'study_id']]
df_bbi_1 = df_bbi_1.drop_duplicates()
df_bbi_counts_1 = df_bbi_1.groupby(["study_id"]).count().reset_index()
df_bbi_counts_1 = df_bbi_counts_1.rename(columns={'calendarDate': 'days'})
display(df_bbi_counts_1)



#### Part 2

In [ ]:
#create dataframes
df_bbi_raw_2 = pd.concat(
    [pd.read_csv(f) for f in bbi_files_2],
    ignore_index=True
)
dt_utc_bbi = pd.to_datetime(
            df_bbi_raw_2["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_bbi_raw_2['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_bbi, df_bbi_raw_2["timezone"])
]


#clean dataframes
##match baseline dates
print(f"BBI data before cropping: {df_bbi_raw_2.shape}")
df_bbi_raw_2 = clean_data_baseline_optimized([df_bbi_raw_2])[0]
print(f"BBI data after cropping: {df_bbi_raw_2.shape}")
display(df_bbi_raw_2.head())
print("------"*20)
##drop days with no data
df_bbi_raw_2 = clean_data_no_data_days_optimized([df_bbi_raw_2])[0]
print(f"BBI data after dropping days with no data: {df_bbi_raw_2.shape}")
display(df_bbi_raw_2.head())
print("------"*20)

#count days per study id
df_bbi_2 = df_bbi_raw_2[['calendarDate', 'study_id']]
df_bbi_2 = df_bbi_2.drop_duplicates()
df_bbi_counts_2 = df_bbi_2.groupby(["study_id"]).count().reset_index()
df_bbi_counts_2 = df_bbi_counts_2.rename(columns={'calendarDate': 'days'})
display(df_bbi_counts_2)



#### Part 3

In [ ]:
#create dataframes
df_bbi_raw_3 = pd.concat(
    [pd.read_csv(f) for f in bbi_files_3],
    ignore_index=True
)
dt_utc_bbi = pd.to_datetime(
            df_bbi_raw_3["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_bbi_raw_3['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_bbi, df_bbi_raw_3["timezone"])
]


#clean dataframes
##match baseline dates
print(f"BBI data before cropping: {df_bbi_raw_3.shape}")
df_bbi_raw_3 = clean_data_baseline_optimized([df_bbi_raw_3])[0]
print(f"BBI data after cropping: {df_bbi_raw_3.shape}")
display(df_bbi_raw_3.head())
print("------"*20)
##drop days with no data
df_bbi_raw_3 = clean_data_no_data_days_optimized([df_bbi_raw_3])[0]
print(f"BBI data after dropping days with no data: {df_bbi_raw_3.shape}")
display(df_bbi_raw_3.head())
print("------"*20)

#count days per study id
df_bbi_3 = df_bbi_raw_3[['calendarDate', 'study_id']]
df_bbi_3 = df_bbi_3.drop_duplicates()
df_bbi_counts_3 = df_bbi_3.groupby(["study_id"]).count().reset_index()
df_bbi_counts_3 = df_bbi_counts_3.rename(columns={'calendarDate': 'days'})
display(df_bbi_counts_3)

#### Part 4

In [ ]:
#create dataframes
df_bbi_raw_4 = pd.concat(
    [pd.read_csv(f) for f in bbi_files_4],
    ignore_index=True
)
dt_utc_bbi = pd.to_datetime(
            df_bbi_raw_4["datetime_utc"],
            errors="coerce",
            utc=True
        )

# Convert each row to its own local timezone, then extract local date
df_bbi_raw_4['calendarDate'] = [
    ts.tz_convert(tz).date() if pd.notna(ts) and pd.notna(tz) else pd.NaT
    for ts, tz in zip(dt_utc_bbi, df_bbi_raw_4["timezone"])
]


#clean dataframes
##match baseline dates
print(f"BBI data before cropping: {df_bbi_raw_4.shape}")
df_bbi_raw_4 = clean_data_baseline_optimized([df_bbi_raw_4])[0]
print(f"BBI data after cropping: {df_bbi_raw_4.shape}")
display(df_bbi_raw_4.head())
print("------"*20)
##drop days with no data
df_bbi_raw_4 = clean_data_no_data_days_optimized([df_bbi_raw_4])[0]
print(f"BBI data after dropping days with no data: {df_bbi_raw_4.shape}")
display(df_bbi_raw_4.head())
print("------"*20)

#count days per study id
df_bbi_4 = df_bbi_raw_4[['calendarDate', 'study_id']]
df_bbi_4 = df_bbi_4.drop_duplicates()
df_bbi_counts_4 = df_bbi_4.groupby(["study_id"]).count().reset_index()
df_bbi_counts_4 = df_bbi_counts_4.rename(columns={'calendarDate': 'days'})
display(df_bbi_counts_4)

In [ ]:
#concat all df_bbi_counts to one df
df_bbi_counts = pd.concat([df_bbi_counts_1, df_bbi_counts_2, df_bbi_counts_3, df_bbi_counts_4], ignore_index=True)

# #add DEC_46 as row with 0 days (as it is missing in the data)
# df_bbi_counts = pd.concat([df_bbi_counts, pd.DataFrame([{'study_id': 'DEC_46', 'days': 0}])], ignore_index=True)
#merge actual with theoretical days
df_device_bbi = df_bbi_counts[['study_id', 'days']].merge(df_visit_dates[['study_id', 'days', 'type']], on='study_id', suffixes=('_actual', '_theoretical'))
display(df_device_bbi)


#total rows theoretically
total_theoretically = df_device_bbi['days_theoretical'].sum()
print("Theoretical rows: ", total_theoretically)

# total actual rows
total_actual = df_device_bbi['days_actual'].sum()
print("Actual rows: ", total_actual)
#number of patientes
n_patients = df_device_bbi['study_id'].nunique()
df_device_bbi_row_counts = pd.DataFrame([{
    'source': 'garmin-device-bbi', 
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients}])
#add to df_row_counts
df_row_counts = pd.concat([df_row_counts, df_device_bbi_row_counts], ignore_index=True)
display(df_row_counts)

#look at study ids where actual != theoretical
df_device_bbi_bad = df_device_bbi[df_device_bbi['days_actual']!= df_device_bbi['days_theoretical']]

#plot theoretical vs actual
fig_f6 = plot_actual_vs_theoretical(
    df_device_bbi_bad,
    title='Garmin Device BBI'
)
fig_f6.show()
    
#store missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('device_bbi_') and file.endswith('.png'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')
    elif file.startswith('garmin-device-bbi_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
fig_f6.write_image(f"../../output/0_missing_data/initial_check/device_bbi_{date}.png", scale=2, width=1200, height=800)
df_device_bbi_bad.to_csv(f'../../output/0_missing_data/initial_check/garmin-device-bbi_{date}.csv', index=False)

## Store summary

In [ ]:
#add total row counts to df_row_counts
total_actual = df_row_counts['actual_rows'].sum()
total_theoretically = df_row_counts['theoretical_rows'].sum()
n_patients = df_row_counts['n_patients'].max()
total_row_counts = pd.DataFrame([{
    'source': 'total',
    'actual_rows': total_actual, 
    'theoretical_rows': total_theoretically, 
    'missing_rows': total_theoretically - total_actual,
    'missing_rows(%)': round((total_theoretically - total_actual) / total_theoretically *100, 1), 
    'n_patients': n_patients
    }])
df_row_counts = pd.concat([df_row_counts, total_row_counts], ignore_index=True)

#store summary of missing data as csv
if not os.path.exists('../../output/0_missing_data/initial_check/archive'):
    os.makedirs('../../output/0_missing_data/initial_check/archive')

files = os.listdir('../../output/0_missing_data/initial_check/')

for file in files:
    if file.startswith('missing_data_summary_') and file.endswith('.csv'):
        os.rename(f'../../output/0_missing_data/initial_check/{file}', f'../../output/0_missing_data/initial_check/archive/{file}')

date = datetime.now().strftime("%Y-%m-%d")
df_row_counts.to_csv(f'../../output/0_missing_data/initial_check/missing_data_summary_{date}.csv', index=False)